# 第4课 模型优化与预测

适合对象：完成第3课并保存了模型的同学

前置知识：
- 会加载模型参数
- 知道模型输出是“每个类别的分数”

学习目标：
- 学会加载训练好的模型
- 学会对单张图片做预测
- 学会分析预测置信度
- 学会测试不同难度图片对结果的影响


## 课程流程

1. 定义模型结构并加载权重
2. 读取测试图像并预测类别
3. 分析 Top-K 置信度
4. 构造不同难度图片并对比
5. 课堂练习


In [ ]:
# 第1步：导入库并定义与第3课一致的模型结构
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageEnhance, ImageFilter

try:
    import torch
    from torch import nn
    from torchvision import transforms
except Exception as e:
    raise ImportError(
        '本课需要 PyTorch 与 torchvision。请先安装：pip install torch torchvision'
    ) from e

class SmallCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('当前设备:', device)


## Step 1 加载模型

重点：模型结构和保存时必须一致，否则参数无法正确加载。


In [ ]:
# 第2步：读取第3课保存的 checkpoint
ckpt_candidates = [
    Path('../models/lesson03_cnn.pth'),
    Path('models/lesson03_cnn.pth'),
]
ckpt_path = next((p for p in ckpt_candidates if p.exists()), None)

class_names = ['类别A', '类别B', '类别C']

if ckpt_path is not None:
    checkpoint = torch.load(ckpt_path, map_location='cpu')
    class_names = checkpoint.get('class_names', class_names)
    model = SmallCNN(num_classes=len(class_names))
    model.load_state_dict(checkpoint['model_state_dict'])
    print('模型加载成功:', ckpt_path.resolve())
else:
    # 找不到模型时也能演示流程，但预测会不准
    print('未找到模型文件，请先运行第3课训练并保存模型。')
    model = SmallCNN(num_classes=len(class_names))

model = model.to(device)
model.eval()

# 预测时要使用和训练一致的预处理
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

print('类别列表:', class_names)


## Step 2 单图预测

流程：
1. 加载图片
2. 预处理
3. 前向推理
4. 用 Softmax 转成概率


In [ ]:
# 第3步：读取一张测试图，做预测
test_candidates = [
    Path('../test.jpg'),
    Path('../result_sketch.jpg'),
    Path('images/sample_building.jpg'),
]
test_path = next((p for p in test_candidates if p.exists()), None)

if test_path is not None:
    test_img = Image.open(test_path).convert('RGB')
    print('测试图片:', test_path)
else:
    # 没有图片时自动构造一张简单图像
    arr = np.zeros((240, 320, 3), dtype=np.uint8)
    arr[..., 2] = np.linspace(255, 120, 240, dtype=np.uint8)[:, None]
    arr[80:220, 90:250, :] = [70, 70, 80]
    test_img = Image.fromarray(arr)
    print('未找到测试图，已自动生成示例图。')

def predict_image(img: Image.Image):
    x = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred_idx = int(np.argmax(probs))
    pred_name = class_names[pred_idx] if pred_idx < len(class_names) else f'类别{pred_idx}'
    pred_conf = float(probs[pred_idx])
    return pred_idx, pred_name, pred_conf, probs

pred_idx, pred_name, pred_conf, probs = predict_image(test_img)
print(f'预测结果: {pred_name} | 置信度: {pred_conf:.2%}')

plt.figure(figsize=(5, 4))
plt.imshow(test_img)
plt.title(f'预测: {pred_name} ({pred_conf:.1%})')
plt.axis('off')
plt.show()


## Step 3 置信度分析

只看“第一名类别”不够，我们还要看：
- 第二名、第三名和第一名差多少？
- 如果第一名概率不高，说明模型不够确定。

一般经验（仅供教学参考）：
- `>=80%`：比较自信
- `50%~80%`：中等
- `<50%`：不太确定，需要人工复核


In [ ]:
# 第4步：画出 Top-K 概率柱状图
top_k = min(5, len(probs))
top_indices = np.argsort(probs)[::-1][:top_k]
top_labels = [class_names[i] if i < len(class_names) else f'类别{i}' for i in top_indices]
top_values = probs[top_indices]

plt.figure(figsize=(8, 4))
bars = plt.bar(top_labels, top_values, color='teal')
plt.ylim(0, 1)
plt.ylabel('概率')
plt.title('Top-K 置信度分析')

for b, v in zip(bars, top_values):
    plt.text(b.get_x() + b.get_width() / 2, v + 0.02, f'{v:.1%}', ha='center', fontsize=9)

plt.show()

if pred_conf >= 0.8:
    print('结论：模型对这张图“比较自信”。')
elif pred_conf >= 0.5:
    print('结论：模型“有一定把握”，建议结合人工观察。')
else:
    print('结论：模型“把握较低”，建议补充训练数据或优化模型。')


## Step 4 测试不同难度图片

我们把同一张图加工成三种难度：
- 简单：原图
- 中等：轻微旋转 + 稍暗
- 困难：灰度 + 模糊 + 低对比度

观察难度变化对置信度的影响。


In [ ]:

# 第5步：构造不同难度，并比较预测结果
easy_img = test_img

# 中等难度：旋转 + 降低亮度
mid_img = test_img.rotate(12)
mid_img = ImageEnhance.Brightness(mid_img).enhance(0.75)

# 困难难度：灰度化 + 模糊 + 低对比度（再转回 RGB 以匹配模型输入）
hard_img = ImageOps.grayscale(test_img)
hard_img = hard_img.filter(ImageFilter.GaussianBlur(radius=1.5))
hard_img = ImageEnhance.Contrast(hard_img).enhance(0.7).convert('RGB')

tests = [('简单', easy_img), ('中等', mid_img), ('困难', hard_img)]

plt.figure(figsize=(12, 4))
for i, (level, img) in enumerate(tests, start=1):
    _, name, conf, _ = predict_image(img)
    plt.subplot(1, 3, i)
    plt.imshow(img)
    plt.title(f'{level}\n{name} ({conf:.1%})')
    plt.axis('off')
    print(f'{level}难度 -> 预测: {name}, 置信度: {conf:.2%}')

plt.tight_layout()
plt.show()


## 课堂练习

1. 再增加一个“超困难”版本（比如强噪声、局部遮挡）。
2. 比较“原图”和“超困难图”的 Top-1、Top-2 概率差值。
3. 思考：模型不自信时，可以做哪些优化？

提示：
- 增加训练数据和数据增强
- 调整学习率和训练轮数
- 使用更强的模型结构


In [ ]:
# 练习答案脚手架：给定概率向量，输出可读的信心报告
def confidence_report(prob_array, class_names):
    prob_array = np.asarray(prob_array)
    order = np.argsort(prob_array)[::-1]
    top1, top2 = order[0], order[1] if len(order) > 1 else order[0]

    name1 = class_names[top1] if top1 < len(class_names) else f'类别{top1}'
    name2 = class_names[top2] if top2 < len(class_names) else f'类别{top2}'
    p1, p2 = float(prob_array[top1]), float(prob_array[top2])

    print(f'Top1: {name1} ({p1:.2%})')
    print(f'Top2: {name2} ({p2:.2%})')
    print(f'领先差值: {(p1 - p2):.2%}')

confidence_report(probs, class_names)
